<span style = "font-family: Verdana; font-size: 20px">

#### **Data Exploration**
</span>

<span style = "font-family: Verdana; font-size: 20px">

Sau các bước tiền xử lý, dữ liệu hiện đã tương đối sạch, nhưng vẫn cần thực hiện khám phá dữ liệu để tiếp tục xử lý những vấn đề khác như cột đa giá trị và dữ liệu trống
</span>

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data/modified/nonempty_pokemon.csv", keep_default_na=False, dtype = {'ID': str})

In [3]:
combats = pd.read_csv("../data/raw_data/combats.csv")

<span style = "font-family: Verdana; font-size: 20px">

#### **I. Pokémon dataset**
</span>

In [4]:
def describe_feature(data, feat, threshold=20):
    
    series = data[feat]
    dtype = series.dtype
    n_unique = series.nunique()
    
    print(f"--- FEATURE: {feat} ---")
    print(f"- Type: {dtype}")
    print(f"- Missing values: {series.isna().sum()} ({series.isna().mean():.2%})")
    print(f"- Distinct values: {n_unique}")
    
    if pd.api.types.is_numeric_dtype(series):
        if n_unique <= threshold:
            print(f"- Distribution (Categorical Numeric):\n{series.value_counts().sort_index()}")
        else:
            print(f"- Statistics:\n{series.describe().round(2)}")
    else:
        if n_unique > threshold:
            print(f"- Too many unique values. Showing TOP {threshold} most frequent:")
            print(series.value_counts().head(threshold))
            print(f"... and {n_unique - threshold} others.")
        else:
            print(f"- Value Counts:\n{series.value_counts()}")
            print(f"- Unique values list: {series.unique()}")
    print("-" * 30 + "\n")
    
def remove_first_last_letter(series, char):
    series = pd.Series([sublist[1:-1] if len(sublist) >= 3 else sublist 
                       for sublist in series])
    return series.str[1:-1].str.split(f"\'{char} \'")

def get_first_element(series):
    return series.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else np.nan)

def get_second_element(series):
    return series.apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else 'None')

In [5]:
print("Pokémon dataset")
print(f"Number of rows: {data.shape[0]}, number of columns: {data.shape[1]}")
print(f"Column names: {data.columns}")
print(f"Number of missing values: {data.isna().sum().sum()}")

Pokémon dataset
Number of rows: 800, number of columns: 48
Column names: Index(['ID', 'Name', 'Type 1', 'Type 2', 'Abilities', 'HiddenAbility',
       'Generation', 'Hp', 'Attack', 'Defense', 'SpecialAttack',
       'SpecialDefense', 'Speed', 'TotalStats', 'Weight', 'Height',
       'GenderProbM', 'Category', 'CatchRate', 'EggCycles', 'EggGroup',
       'LevelingRate', 'BaseFriendship', 'IsLegendary', 'IsMythical',
       'IsUltraBeast', 'HasMega', 'EvoStage', 'TotalEvoStages', 'PreevoName',
       'DamageFromNormal', 'DamageFromFighting', 'DamageFromFlying',
       'DamageFromPoison', 'DamageFromGround', 'DamageFromRock',
       'DamageFromBug', 'DamageFromGhost', 'DamageFromSteel', 'DamageFromFire',
       'DamageFromWater', 'DamageFromGrass', 'DamageFromElectric',
       'DamageFromPsychic', 'DamageFromIce', 'DamageFromDragon',
       'DamageFromDark', 'DamageFromFairy'],
      dtype='object')
Number of missing values: 0


In [6]:
pd.options.display.max_columns = None
data.head(5)

,ID,Name,Type 1,Type 2,Abilities,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,EggGroup,LevelingRate,BaseFriendship,IsLegendary,IsMythical,IsUltraBeast,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy
0,1,Bulbasaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,45.0,49.0,49.0,65.0,65.0,45.0,318.0,6.9,0.7,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,0,1,3,No Preevolution,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,60.0,62.0,63.0,80.0,80.0,60.0,405.0,13.0,1.0,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,0,2,3,Bulbasaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
2,3,Venusaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,80.0,82.0,83.0,100.0,100.0,80.0,625.0,100.0,2.0,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,1,3,3,Ivysaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
3,4,Mega Venusaur,Grass,Poison,['Thick Fat'],[],I,80.0,100.0,123.0,122.0,120.0,80.0,625.0,155.5,2.4,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,1,3,3,Venusaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.5,2.0,1.0,1.0,1.0,0.5
4,5,Charmander,Fire,,['Blaze'],['Solar Power'],I,39.0,52.0,43.0,60.0,50.0,65.0,309.0,8.5,0.6,0.875,Lizard Pokémon,45,20,['Monster' 'Dragon'],Medium Slow,70,0,0,0,0,1,3,No Preevolution,1.0,1.0,1.0,1.0,2.0,2.0,0.5,1.0,0.5,0.5,2.0,0.50,1.0,1.0,0.5,1.0,1.0,0.5



##### **ID**


In [7]:
describe_feature(data, 'ID')

--- FEATURE: ID ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 800
- Too many unique values. Showing TOP 20 most frequent:
ID
1      1
538    1
528    1
529    1
530    1
531    1
532    1
533    1
534    1
535    1
536    1
537    1
539    1
2      1
540    1
541    1
542    1
543    1
544    1
545    1
Name: count, dtype: int64
... and 780 others.
------------------------------





Mỗi Pokémon có một số ID riêng biệt duy nhất, các dạng tiến hoá cũng được xem như một Pokémon riêng biệt với số ID riêng.


##### **Name**


In [8]:
describe_feature(data, 'Name')

--- FEATURE: Name ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 800
- Too many unique values. Showing TOP 20 most frequent:
Name
Bulbasaur                 1
Uxie                      1
Mega Gallade              1
Probopass                 1
Dusknoir                  1
Froslass                  1
Rotom                     1
Heat Rotom                1
Wash Rotom                1
Frost Rotom               1
Fan Rotom                 1
Mow Rotom                 1
Mesprit                   1
Ivysaur                   1
Azelf                     1
Dialga                    1
Palkia                    1
Heatran                   1
Regigigas                 1
Giratina Altered Forme    1
Name: count, dtype: int64
... and 780 others.
------------------------------





Mỗi Pokémon cũng có tên riêng, một trong hai cột này đều có thể được dùng để làm primary key.


##### **Type**
</span>



Để thuận tiện cho việc phân tích, nếu Pokémon chỉ có một loại thì ta sẽ điền giá trị của `Type 1` vào `Type 2`.

In [9]:
data.rename(columns={'Type 1': 'Type1', 'Type 2': 'Type2'}, inplace=True)
describe_feature(data, 'Type1')
describe_feature(data, 'Type2')
data.loc[data['Type2'] == "None", 'Type2'] = data['Type1']

--- FEATURE: Type1 ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 18
- Value Counts:
Type1
Water       112
Normal       98
Grass        70
Bug          69
Psychic      57
Fire         52
Electric     44
Rock         44
Dragon       32
Ground       32
Ghost        32
Dark         31
Poison       28
Steel        27
Fighting     27
Ice          24
Fairy        17
Flying        4
Name: count, dtype: int64
- Unique values list: ['Grass' 'Fire' 'Water' 'Bug' 'Normal' 'Poison' 'Electric' 'Ground'
 'Fairy' 'Fighting' 'Psychic' 'Rock' 'Ghost' 'Ice' 'Dragon' 'Dark' 'Steel'
 'Flying']
------------------------------

--- FEATURE: Type2 ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 19
- Value Counts:
Type2
            386
Flying       97
Ground       35
Poison       34
Psychic      33
Fighting     26
Grass        25
Fairy        23
Steel        22
Dark         20
Dragon       18
Ice          14
Rock         14
Water        14
Ghost        14
Fire         12
El



Thuộc tính của mỗi Pokémon được phân thành hai loại chính và phụ, tuy nhiên thứ tự của hai thuộc tính này không quan trọng.



##### **Abilities**
</span>

In [10]:
for col in ['Abilities', 'HiddenAbility']:
    for abi in data[col].head(10):
        print(abi)

['Overgrow']
['Overgrow']
['Overgrow']
['Thick Fat']
['Blaze']
['Blaze']
['Blaze']
['Tough Claws']
['Drought']
['Torrent']
['Chlorophyll']
['Chlorophyll']
['Chlorophyll']
[]
['Solar Power']
['Solar Power']
['Solar Power']
[]
[]
['Rain Dish']




Thuộc tính này có chứa một hoặc nhiều thành phần, bao gồm cả các ký tự đặc biệt, cần phải được làm sạch và tách riêng.

In [11]:
data['Abilities'] = remove_first_last_letter(data['Abilities'], ',')
data['HiddenAbility'] = remove_first_last_letter(data['HiddenAbility'], ',')
data['Ability1'] = get_first_element(data['Abilities'])
data['Ability2'] = get_second_element(data['Abilities'])
data['HiddenAbility'] = get_first_element(data['HiddenAbility'])
data = data.drop('Abilities', axis=1)

In [12]:
describe_feature(data, 'Ability1')

--- FEATURE: Ability1 ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 149
- Too many unique values. Showing TOP 20 most frequent:
Ability1
Levitate        39
Swift Swim      28
Chlorophyll     25
Pressure        23
Intimidate      22
Keen Eye        20
Sturdy          19
Overgrow        18
Torrent         18
Blaze           18
Swarm           16
Thick Fat       15
Run Away        14
Shed Skin       13
Poison Point    13
Frisk           13
Natural Cure    12
Cute Charm      12
Static          12
Oblivious       12
Name: count, dtype: int64
... and 129 others.
------------------------------



In [13]:
describe_feature(data, 'Ability2')

--- FEATURE: Ability2 ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 112
- Too many unique values. Showing TOP 20 most frequent:
Ability2
None            392
Shell Armor      11
Sturdy           11
Inner Focus      10
Sniper            9
Hydration         9
Own Tempo         9
Early Bird        9
Technician        8
Rock Head         8
Infiltrator       7
Keen Eye          7
Flash Fire        7
Ice Body          7
Competitive       7
Magic Guard       7
Super Luck        6
Serene Grace      6
Sticky Hold       6
Rivalry           6
Name: count, dtype: int64
... and 92 others.
------------------------------





Thay thế các ô bị trống bằng `None`

In [14]:
data['Ability1'] = data['Ability1'].replace(['nan', ''], ['None', 'None'])
data['Ability2'] = data['Ability2'].replace(['nan', ''], ['None', 'None'])
data['HiddenAbility'] = data['HiddenAbility'].replace(['nan', ''], ['None', 'None'])
data['Ability1'] = data['Ability1'].fillna('None')
data['Ability2'] = data['Ability2'].fillna('None')
data['HiddenAbility'] = data['HiddenAbility'].fillna('None')

In [15]:
describe_feature(data, 'HiddenAbility')

--- FEATURE: HiddenAbility ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 135
- Too many unique values. Showing TOP 20 most frequent:
HiddenAbility
None           155
Sheer Force     17
Unnerve         15
Weak Armor      15
Telepathy       14
Infiltrator     14
Overcoat        14
Regenerator     13
Analytic        12
Rattled         11
Sand Force      11
Swift Swim      10
Damp            10
Hydration        9
Gluttony         9
Inner Focus      9
Insomnia         9
Moxie            8
Sap Sipper       8
Reckless         8
Name: count, dtype: int64
... and 115 others.
------------------------------





##### **Generation**
</span>

In [16]:
describe_feature(data, 'Generation')

--- FEATURE: Generation ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 6
- Value Counts:
Generation
I      166
V      165
III    160
IV     121
II     106
VI      82
Name: count, dtype: int64
- Unique values list: ['I' 'II' 'III' 'IV' 'V' 'VI']
------------------------------



In [17]:
data['Generation'] = data['Generation'].replace(['I', 'II', 'III', 'IV', 'V', 'VI'], [1, 2, 3, 4, 5, 6]).infer_objects(copy=False)

C:\Users\HP\AppData\Local\Temp\ipykernel_18852\1242375916.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Generation'] = data['Generation'].replace(['I', 'II', 'III', 'IV', 'V', 'VI'], [1, 2, 3, 4, 5, 6]).infer_objects(copy=False)


In [18]:
describe_feature(data, 'Generation')

--- FEATURE: Generation ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
Generation
1    166
2    106
3    160
4    121
5    165
6     82
Name: count, dtype: int64
------------------------------





##### **HP**
</span>

In [19]:
describe_feature(data, 'Hp')

--- FEATURE: Hp ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 94
- Statistics:
count    800.00
mean      69.42
std       25.54
min        1.00
25%       50.00
50%       65.00
75%       80.00
max      255.00
Name: Hp, dtype: float64
------------------------------





##### **Attack**
</span>

In [20]:
describe_feature(data, 'Attack')

--- FEATURE: Attack ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 111
- Statistics:
count    800.00
mean      79.16
std       32.46
min        5.00
25%       55.00
50%       75.00
75%      100.00
max      190.00
Name: Attack, dtype: float64
------------------------------





##### **Defense**
</span>

In [21]:
describe_feature(data, 'Defense')

--- FEATURE: Defense ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 103
- Statistics:
count    800.00
mean      73.96
std       31.09
min        5.00
25%       50.00
50%       70.00
75%       90.00
max      230.00
Name: Defense, dtype: float64
------------------------------





##### **SpecialAttack**
</span>

In [22]:
describe_feature(data, 'SpecialAttack')

--- FEATURE: SpecialAttack ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 105
- Statistics:
count    800.00
mean      72.96
std       32.76
min       10.00
25%       49.75
50%       65.00
75%       95.00
max      194.00
Name: SpecialAttack, dtype: float64
------------------------------





##### **SpecialDefense**
</span>

In [23]:
describe_feature(data, 'SpecialDefense')

--- FEATURE: SpecialDefense ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 92
- Statistics:
count    800.00
mean      72.05
std       27.90
min       20.00
25%       50.00
50%       70.00
75%       90.00
max      230.00
Name: SpecialDefense, dtype: float64
------------------------------





##### **Speed**
</span>

In [24]:
describe_feature(data, 'Speed')

--- FEATURE: Speed ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 108
- Statistics:
count    800.00
mean      68.40
std       29.13
min        5.00
25%       45.00
50%       65.00
75%       90.00
max      180.00
Name: Speed, dtype: float64
------------------------------





##### **TotalStats**
</span>

In [25]:
describe_feature(data, 'TotalStats')

--- FEATURE: TotalStats ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 196
- Statistics:
count    800.00
mean     442.70
std      126.87
min      180.00
25%      330.00
50%      455.00
75%      525.00
max      780.00
Name: TotalStats, dtype: float64
------------------------------



In [26]:
for i in range(data.shape[0]):
    total = data.loc[i, 'Hp'] + data.loc[i, 'Attack'] + data.loc[i, 'Defense'] + data.loc[i, 'SpecialAttack'] + data.loc[i, 'SpecialDefense'] + data.loc[i, 'Speed'] # type: ignore
    if not data.at[i, 'TotalStats'] == total:
        print(data.at[i, 'Name'], data.at[i, 'TotalStats'],  total)

Venusaur 625.0 525.0
Charizard 634.0 534.0
Blastoise 630.0 530.0
Beedrill 495.0 395.0
Pidgeot 579.0 479.0
Pikachu 430.0 320.0
Alakazam 600.0 500.0
Gengar 600.0 500.0
Kangaskhan 590.0 490.0
Pinsir 600.0 500.0
Gyarados 640.0 540.0
Eevee 435.0 325.0
Aerodactyl 615.0 515.0
Mewtwo 780.0 680.0
Ampharos 610.0 510.0
Steelix 610.0 510.0
Scizor 600.0 500.0
Heracross 600.0 500.0
Houndoom 600.0 500.0
Tyranitar 700.0 600.0
Sceptile 630.0 530.0
Blaziken 630.0 530.0
Swampert 635.0 535.0
Gardevoir 618.0 518.0
Sableye 480.0 380.0
Mawile 480.0 380.0
Aggron 630.0 530.0
Medicham 510.0 410.0
Manectric 575.0 475.0
Sharpedo 560.0 460.0
Camerupt 560.0 460.0
Altaria 590.0 490.0
Banette 555.0 455.0
Absol 565.0 465.0
Glalie 580.0 480.0
Salamence 700.0 600.0
Metagross 700.0 600.0
Latias 700.0 600.0
Latios 700.0 600.0
Kyogre 770.0 670.0
Groudon 770.0 670.0
Rayquaza 780.0 680.0
Lopunny 580.0 480.0
Garchomp 700.0 600.0
Lucario 625.0 525.0
Abomasnow 594.0 494.0
Gallade 618.0 518.0
Rotom 520.0 440.0
Audino 545.0 445.0



Có vẻ như tổng điểm chỉ số của một số các Pokémon có dạng tiến hoá Mega bị lệch so với tổng chỉ số thực tế của chúng, cần phải tính lại cột này.

In [27]:
data['TotalStats'] = data['Hp'] + data['Attack'] + data['Defense'] + data['SpecialAttack'] + data['SpecialDefense'] + data['Speed']



##### **Weight**
</span>

In [28]:
describe_feature(data, 'Weight')

--- FEATURE: Weight ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 421
- Statistics:
count    800.00
mean      63.93
std      107.99
min        0.10
25%        9.80
50%       29.95
75%       65.78
max      999.70
Name: Weight, dtype: float64
------------------------------





##### **Height**
</span>

In [29]:
describe_feature(data, 'Height')

--- FEATURE: Height ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 51
- Statistics:
count    800.00
mean       1.22
std        1.18
min        0.10
25%        0.60
50%        1.00
75%        1.50
max       14.50
Name: Height, dtype: float64
------------------------------





##### **GenderProbM**
</span>

In [30]:
describe_feature(data, 'GenderProbM')

--- FEATURE: GenderProbM ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 9
- Value Counts:
GenderProbM
0.5      495
0.875    110
-         92
0.0       28
1.0       24
0.25      22
0.75      20
-1         7
0.125      2
Name: count, dtype: int64
- Unique values list: ['0.875' '0.5' '0.0' '1.0' '0.25' '0.75' '-' '-1' '0.125']
------------------------------



In [31]:
data['GenderProbM'].unique()

array(['0.875', '0.5', '0.0', '1.0', '0.25', '0.75', '-', '-1', '0.125'],
      dtype=object)



GenderProbM cho biết tỷ lệ giống đực của Pokémon, giá trị `-1` biểu thị cho các Pokémon không có giới tính như các Pokémon huyền thoại hay thần thoại. Để khoảng giá trị của cột này liên tục từ 0 đến 1, ta sẽ thay thế các giá trị `-1` này bằng `0.5`. Giá trị `-` cũng được thay thế bằng `0.0`

In [32]:
replace_rules = {'-': 0.0, -1: 0.5}
data['GenderProbM'] = data['GenderProbM'].replace(replace_rules)
data['GenderProbM'] = pd.to_numeric(data['GenderProbM'], errors='coerce')
describe_feature(data, 'GenderProbM')

--- FEATURE: GenderProbM ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 8
- Distribution (Categorical Numeric):
GenderProbM
-1.000      7
 0.000    120
 0.125      2
 0.250     22
 0.500    495
 0.750     20
 0.875    110
 1.000     24
Name: count, dtype: int64
------------------------------





##### **Category**
</span>

In [33]:
describe_feature(data, 'Category')

--- FEATURE: Category ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 524
- Too many unique values. Showing TOP 20 most frequent:
Category
Dragon Pokémon        9
Pumpkin Pokémon       8
Flame Pokémon         7
Mushroom Pokémon      6
Bagworm Pokémon       6
Mouse Pokémon         6
Plasma Pokémon        6
Fox Pokémon           5
Balloon Pokémon       5
Seed Pokémon          5
Bat Pokémon           4
Poison Pin Pokémon    4
Mud Fish Pokémon      4
Iron Armor Pokémon    4
Eon Pokémon           4
Fairy Pokémon         4
Psi Pokémon           4
Drill Pokémon         4
Tadpole Pokémon       4
DNA Pokémon           4
Name: count, dtype: int64
... and 504 others.
------------------------------




##### **CatchRate**
</span>

In [34]:
describe_feature(data, 'CatchRate')

--- FEATURE: CatchRate ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 33
- Statistics:
count    800.00
mean      94.76
std       75.72
min        3.00
25%       45.00
50%       60.00
75%      140.00
max      255.00
Name: CatchRate, dtype: float64
------------------------------





##### **EggCycles**
</span>

In [35]:
describe_feature(data, 'EggCycles')

--- FEATURE: EggCycles ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 10
- Distribution (Categorical Numeric):
EggCycles
5        3
10      21
15     112
20     449
25      61
30      26
35      15
40      43
80      17
120     53
Name: count, dtype: int64
------------------------------





##### **EggGroup**
</span>

In [36]:
describe_feature(data, 'EggGroup')

--- FEATURE: EggGroup ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 60
- Too many unique values. Showing TOP 20 most frequent:
EggGroup
['Field']                 141
-                          86
['Bug']                    60
['Amorphous']              50
['Mineral']                46
['Human-Like']             39
['Flying']                 36
['Monster']                21
['Monster' 'Dragon']       20
['Grass']                  19
['Fairy']                  17
['Water 1' 'Field']        17
['Monster' 'Water 1']      16
['Water 3']                14
['Monster' 'Grass']        14
['Water 1']                14
['Monster' 'Field']        14
['Field' 'Human-Like']     13
['Water 2']                12
['Water 1' 'Water 3']      11
Name: count, dtype: int64
... and 40 others.
------------------------------



Cột `EggGroup` gặp vấn đề tương tự cột Type, tuy nhiên không có ảnh hưởng đến việc dự đoán Pokemon chiến thắng nên có thể drop

In [37]:
data.drop(columns = ['EggGroup'], inplace=True)



##### **Leveling Rate**
</span>

In [38]:
describe_feature(data, 'LevelingRate')

--- FEATURE: LevelingRate ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 6
- Value Counts:
LevelingRate
Medium Fast    320
Medium Slow    205
Slow           182
Fast            56
Erratic         23
Fluctuating     14
Name: count, dtype: int64
- Unique values list: ['Medium Slow' 'Medium Fast' 'Fast' 'Slow' 'Fluctuating' 'Erratic']
------------------------------





##### **Base Frienship**
</span>

In [39]:
describe_feature(data, 'BaseFriendship')

--- FEATURE: BaseFriendship ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
BaseFriendship
0       33
35      80
70     649
90      10
100     17
140     11
Name: count, dtype: int64
------------------------------





##### **IsLegendary**
</span>

In [40]:
describe_feature(data, 'IsLegendary')

--- FEATURE: IsLegendary ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 2
- Distribution (Categorical Numeric):
IsLegendary
0    749
1     51
Name: count, dtype: int64
------------------------------





##### **IsMythical**
</span>

In [41]:
describe_feature(data, 'IsMythical')

--- FEATURE: IsMythical ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 2
- Distribution (Categorical Numeric):
IsMythical
0    776
1     24
Name: count, dtype: int64
------------------------------





##### **IsUltraBeast**
</span>

In [42]:
describe_feature(data, 'IsUltraBeast')

--- FEATURE: IsUltraBeast ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 1
- Distribution (Categorical Numeric):
IsUltraBeast
0    800
Name: count, dtype: int64
------------------------------





Có vẻ là không có Pokémon siêu thú nào trong bộ dữ liệu này, nên cột này có thể bị loại bỏ.

In [43]:
data = data.drop('IsUltraBeast', axis=1)



##### **HasMega**
</span>

In [44]:
describe_feature(data, 'HasMega')

--- FEATURE: HasMega ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 2
- Distribution (Categorical Numeric):
HasMega
0    696
1    104
Name: count, dtype: int64
------------------------------





##### **EvoStage**
</span>

In [45]:
describe_feature(data, 'EvoStage')

--- FEATURE: EvoStage ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 3
- Distribution (Categorical Numeric):
EvoStage
1    403
2    287
3    110
Name: count, dtype: int64
------------------------------





Cột này cho biết bậc tiến hoá hiện tại của mỗi Pokémon, giá trị có thể có là `1` (cho bậc tiến hoá đầu tiên), lần lượt là `2` và `3`. Pokémon không có tiến hoá sẽ có giá trị là `1`. Vấn đề có thể có là EvoStage có giá trị lớn hơn TotalEvoStages.

In [46]:
data.loc[data['EvoStage'] > data['TotalEvoStages']]

,ID,Name,Type1,Type2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability1,Ability2
316,317,Shedinja,Bug,Ghost,None,3,1.0,90.0,45.0,30.0,30.0,40.0,236.0,1.2,0.8,0.0,Shed Pokémon,45,15,Erratic,70,0,0,0,3,2,Ninjask,0.0,0.0,2.0,0.5,0.5,2.0,0.5,2.0,1.0,2.0,1.0,0.5,1.0,1.0,1.0,1.0,2.0,1.0,Wonder Guard,None


In [47]:
data.loc[data['EvoStage'] > data['TotalEvoStages'], 'EvoStage'] = data['TotalEvoStages']
data.loc[data['EvoStage'] > data['TotalEvoStages']]

,ID,Name,Type1,Type2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability1,Ability2




##### **TotalEvoStages**
</span>

In [48]:
describe_feature(data, 'TotalEvoStages')

--- FEATURE: TotalEvoStages ---
- Type: int64
- Missing values: 0 (0.00%)
- Distinct values: 3
- Distribution (Categorical Numeric):
TotalEvoStages
1    144
2    370
3    286
Name: count, dtype: int64
------------------------------





##### **PreevoName**
</span>

In [49]:
describe_feature(data, 'PreevoName')

--- FEATURE: PreevoName ---
- Type: object
- Missing values: 0 (0.00%)
- Distinct values: 393
- Too many unique values. Showing TOP 20 most frequent:
PreevoName
No Preevolution    387
Eevee                8
Tyrogue              3
Mewtwo               2
Wurmple              2
Kirlia               2
Poliwhirl            2
Gloom                2
Darumaka             2
Espurr               2
Doublade             2
Slowpoke             2
Snorunt              2
Clamperl             2
Charizard            2
Dewott               1
Pansear              1
Magneton             1
Sneasel              1
Abomasnow            1
Name: count, dtype: int64
... and 373 others.
------------------------------





##### **DamageFrom...**
</span>

In [50]:
describe_feature(data, 'DamageFromNormal')

--- FEATURE: DamageFromNormal ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 4
- Distribution (Categorical Numeric):
DamageFromNormal
0.00     46
0.25      6
0.50     91
1.00    657
Name: count, dtype: int64
------------------------------



In [51]:
describe_feature(data, 'DamageFromFighting')

--- FEATURE: DamageFromFighting ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 7
- Distribution (Categorical Numeric):
DamageFromFighting
0.00     46
0.25     44
0.50    192
1.00    321
1.50      1
2.00    182
4.00     14
Name: count, dtype: int64
------------------------------



In [52]:
describe_feature(data, 'DamageFromFlying')

--- FEATURE: DamageFromFlying ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromFlying
0.25      9
0.50    116
1.00    489
2.00    175
4.00     11
Name: count, dtype: int64
------------------------------



In [53]:
describe_feature(data, 'DamageFromPoison')

--- FEATURE: DamageFromPoison ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
DamageFromPoison
0.00     49
0.25     17
0.50    155
1.00    482
2.00     95
4.00      2
Name: count, dtype: int64
------------------------------



In [54]:
describe_feature(data, 'DamageFromGround')

--- FEATURE: DamageFromGround ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 7
- Distribution (Categorical Numeric):
DamageFromGround
0.00    104
0.25      6
0.50     87
1.00    400
1.50      1
2.00    190
4.00     12
Name: count, dtype: int64
------------------------------



In [55]:
describe_feature(data, 'DamageFromRock')

--- FEATURE: DamageFromRock ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromRock
0.25      5
0.50    126
1.00    449
2.00    196
4.00     24
Name: count, dtype: int64
------------------------------



In [56]:
describe_feature(data, 'DamageFromBug')

--- FEATURE: DamageFromBug ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromBug
0.25     41
0.50    248
1.00    370
2.00    132
4.00      9
Name: count, dtype: int64
------------------------------



In [57]:
describe_feature(data, 'DamageFromGhost')

--- FEATURE: DamageFromGhost ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromGhost
0.0    102
0.5     44
1.0    527
2.0    126
4.0      1
Name: count, dtype: int64
------------------------------



In [58]:
describe_feature(data, 'DamageFromSteel')

--- FEATURE: DamageFromSteel ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromSteel
0.25     10
0.50    239
1.00    450
2.00     96
4.00      5
Name: count, dtype: int64
------------------------------



In [59]:
describe_feature(data, 'DamageFromFire')

--- FEATURE: DamageFromFire ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
DamageFromFire
0.25     18
0.50    228
1.00    353
1.50      1
2.00    182
4.00     18
Name: count, dtype: int64
------------------------------



In [60]:
describe_feature(data, 'DamageFromWater')

--- FEATURE: DamageFromWater ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromWater
0.25      6
0.50    224
1.00    430
2.00    126
4.00     14
Name: count, dtype: int64
------------------------------



In [61]:
describe_feature(data, 'DamageFromGrass')

--- FEATURE: DamageFromGrass ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromGrass
0.25     88
0.50    255
1.00    298
2.00    130
4.00     29
Name: count, dtype: int64
------------------------------



In [62]:
describe_feature(data, 'DamageFromElectric')

--- FEATURE: DamageFromElectric ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
DamageFromElectric
0.00     68
0.25      3
0.50    152
1.00    397
2.00    172
4.00      8
Name: count, dtype: int64
------------------------------



In [63]:
describe_feature(data, 'DamageFromPsychic')

--- FEATURE: DamageFromPsychic ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 6
- Distribution (Categorical Numeric):
DamageFromPsychic
0.00     50
0.25      7
0.50    111
1.00    534
2.00     96
4.00      2
Name: count, dtype: int64
------------------------------



In [64]:
describe_feature(data, 'DamageFromIce')

--- FEATURE: DamageFromIce ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromIce
0.25      9
0.50    206
1.00    351
2.00    208
4.00     26
Name: count, dtype: int64
------------------------------



In [65]:
describe_feature(data, 'DamageFromDragon')

--- FEATURE: DamageFromDragon ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 4
- Distribution (Categorical Numeric):
DamageFromDragon
0.0     40
0.5     45
1.0    667
2.0     48
Name: count, dtype: int64
------------------------------



In [66]:
describe_feature(data, 'DamageFromDark')

--- FEATURE: DamageFromDark ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromDark
0.25      3
0.50    119
1.00    561
2.00    116
4.00      1
Name: count, dtype: int64
------------------------------



In [67]:
describe_feature(data, 'DamageFromFairy')

--- FEATURE: DamageFromFairy ---
- Type: float64
- Missing values: 0 (0.00%)
- Distinct values: 5
- Distribution (Categorical Numeric):
DamageFromFairy
0.25      1
0.50    149
1.00    527
2.00    117
4.00      6
Name: count, dtype: int64
------------------------------



<span style = "font-family: Verdana; font-size: 20px">

#### **II. Combat dataset**
</span>

In [68]:
print("Combat dataset")
print(f"Number of rows: {combats.shape[0]}, number of columns: {combats.shape[1]}")
print(f"Column names: {combats.columns}")
print(f"Number of missing values: {combats.isna().sum().sum()}")

Combat dataset
Number of rows: 50000, number of columns: 3
Column names: Index(['First_pokemon', 'Second_pokemon', 'Winner'], dtype='object')
Number of missing values: 0


In [69]:
# Tổng số trận thắng của mỗi Pokémon
total_wins = combats['Winner'].value_counts()
# Số trận thắng của mỗi Pokémon
number_of_wins = combats.groupby('Winner').count()

countByFirst = combats.groupby('Second_pokemon').count() # 
countBySecond = combats.groupby('First_pokemon').count()
print("Looking at the dimensions of our dataframes")
print("Count by first winner shape: " + str(countByFirst.shape))
print("Count by second winner shape: " + str(countBySecond.shape))
print("Total wins shape : " + str(total_wins.shape))

Looking at the dimensions of our dataframes
Count by first winner shape: (784, 2)
Count by second winner shape: (784, 2)
Total wins shape : (783,)




Có thể thấy số chiều của dataframe tổng trận thắng không giống với số chiều của hai dataframe đếm số trận thắng theo từng Pokémon. 

Điều này cho thấy có một Pokémon chưa từng thắng trận nào trong dữ liệu.

In [70]:
locate_losing_pokemon = np.setdiff1d(countByFirst.index.values, number_of_wins.index.values)-1
losing_pokemon = data.iloc[locate_losing_pokemon[0],]
losing_pokemon

ID                                231
Name                          Shuckle
Type1                             Bug
Type2                            Rock
HiddenAbility                Contrary
Generation                          2
Hp                               20.0
Attack                           10.0
Defense                         230.0
SpecialAttack                    10.0
SpecialDefense                  230.0
Speed                             5.0
TotalStats                      505.0
Weight                           20.5
Height                            0.6
GenderProbM                       0.5
Category                 Mold Pokémon
CatchRate                         190
EggCycles                          20
LevelingRate              Medium Slow
BaseFriendship                     70
IsLegendary                         0
IsMythical                          0
HasMega                             0
EvoStage                            1
TotalEvoStages                      1
PreevoName  



Tội nghiệp Shuckle 🙁 Tuy chỉ số phòng thủ khá ấn tượng, cao nhất trong số các Pokémon, nhưng các chỉ số khác lại cực kỳ thấp so với mặt bằng chung.



Có tồn tại Pokémon chưa từng thắng trận nào, vậy chắc cũng phải có Pokémon nào chưa từng tham gia trận nào chứ?

In [71]:
pokemon_not_in_combat = data[~data['ID'].astype(int).isin(number_of_wins.index)]
pokemon_not_in_combat

,ID,Name,Type1,Type2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability1,Ability2
11,12,Blastoise,Water,,Rain Dish,1,79.0,83.0,100.0,85.0,105.0,78.0,530.0,85.5,1.6,0.875,Shellfish Pokémon,45,20,Medium Slow,70,0,0,1,3,3,Wartortle,1.0,1.00,1.00,1.0,1.0,1.0,1.0,1.0,0.50,0.5,0.5,2.00,2.0,1.0,0.5,1.0,1.0,1.0,Torrent,None
32,33,Sandshrew,Ground,,Sand Rush,1,50.0,75.0,85.0,20.0,30.0,40.0,300.0,12.0,0.6,0.500,Mouse Pokémon,255,20,Medium Fast,70,0,0,0,1,2,No Preevolution,1.0,1.00,1.00,0.5,1.0,0.5,1.0,1.0,1.00,1.0,2.0,2.00,0.0,1.0,2.0,1.0,1.0,1.0,Sand Veil,None
45,46,Wigglytuff,Normal,Fairy,Frisk,1,140.0,70.0,45.0,85.0,50.0,45.0,435.0,12.0,1.0,0.250,Balloon Pokémon,50,10,Fast,70,0,0,0,3,3,Jigglypuff,1.0,1.00,1.00,2.0,1.0,1.0,0.5,0.0,2.00,1.0,1.0,1.00,1.0,1.0,1.0,0.0,0.5,1.0,Cute Charm,Competitive
65,66,Poliwag,Water,,Swift Swim,1,40.0,50.0,40.0,40.0,40.0,90.0,300.0,12.4,0.6,0.500,Tadpole Pokémon,255,20,Medium Slow,70,0,0,0,1,3,No Preevolution,1.0,1.00,1.00,1.0,1.0,1.0,1.0,1.0,0.50,0.5,0.5,2.00,2.0,1.0,0.5,1.0,1.0,1.0,Water Absorb,Damp
77,78,Victreebel,Grass,Poison,Gluttony,1,80.0,105.0,65.0,100.0,70.0,70.0,490.0,15.5,1.7,0.500,Flycatcher Pokémon,45,20,Medium Slow,70,0,0,0,3,3,Weepinbell,1.0,0.50,2.00,1.0,1.0,1.0,1.0,1.0,1.00,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5,Chlorophyll,None
89,90,Magneton,Electric,Steel,Analytic,1,50.0,60.0,95.0,120.0,70.0,70.0,465.0,60.0,1.0,0.000,Magnet Pokémon,60,20,Medium Fast,70,0,0,0,2,3,Magnemite,0.5,2.00,0.25,0.0,4.0,0.5,0.5,1.0,0.25,2.0,1.0,0.50,0.5,0.5,0.5,0.5,1.0,0.5,Magnet Pull,Sturdy
143,144,Ditto,Normal,,Imposter,1,48.0,48.0,48.0,48.0,48.0,48.0,288.0,4.0,0.3,0.000,Transform Pokémon,35,20,Medium Fast,70,0,0,0,1,1,No Preevolution,1.0,2.00,1.00,1.0,1.0,1.0,1.0,0.0,1.00,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Limber,None
182,183,Ariados,Bug,Poison,Sniper,2,70.0,90.0,70.0,60.0,70.0,40.0,400.0,33.5,1.1,0.500,Long Leg Pokémon,90,15,Fast,70,0,0,0,2,2,Spinarak,1.0,0.25,2.00,0.5,1.0,2.0,0.5,1.0,1.00,2.0,1.0,0.25,1.0,2.0,1.0,1.0,1.0,0.5,Swarm,Insomnia
230,231,Shuckle,Bug,Rock,Contrary,2,20.0,10.0,230.0,10.0,230.0,5.0,505.0,20.5,0.6,0.500,Mold Pokémon,190,20,Medium Slow,70,0,0,0,1,1,No Preevolution,0.5,1.00,1.00,0.5,1.0,2.0,1.0,1.0,2.00,1.0,2.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Sturdy,Gluttony
235,236,Ursaring,Normal,,Unnerve,2,90.0,130.0,75.0,75.0,75.0,55.0,500.0,125.8,1.8,0.500,Hibernator Pokémon,60,20,Medium Fast,70,0,0,0,2,3,Teddiursa,1.0,2.00,1.00,1.0,1.0,1.0,1.0,0.0,1.00,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Guts,Quick Feet




Vậy linh cảm trước đó là đúng, thực sự có Pokémon chưa từng tham gia trận nào.



Tính toán tỷ lệ thắng `WinRate`, tổng trận tham gia `TotalFights` và trận thắng `FightsWon` cho mỗi Pokémon, sẽ rất hữu ích cho giai đoạn trực quan hoá sắp tới.
</span>

In [72]:
number_of_wins = number_of_wins.sort_index()
number_of_wins.rename(columns={'First_pokemon': 'FightsWon'}, inplace=True)
number_of_wins = number_of_wins.drop('Second_pokemon', axis = 1)
number_of_wins['TotalFights'] = countByFirst.Winner + countBySecond.Winner

In [73]:
number_of_wins.info()

<class 'pandas.core.frame.DataFrame'>
Index: 783 entries, 1 to 800
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   FightsWon    783 non-null    int64
 1   TotalFights  783 non-null    int64
dtypes: int64(2)
memory usage: 18.4 KB


In [74]:
data['ID'] = data['ID'].astype(int) 

In [75]:
data = pd.merge(data, number_of_wins, left_on='ID', right_index = True, how='left')



Xử lý định dạng và missing value cho các cột mới được thêm. 

In [76]:
data['FightsWon'] = data['FightsWon'].fillna(0).astype(int)
data['TotalFights'] = data['TotalFights'].fillna(0).astype(int)
data['WinPercentage'] = (data['FightsWon']/data['TotalFights']).round(6)
data['WinPercentage'] = data['WinPercentage'].fillna(0.0)

In [78]:
with open('../data/modified/viz_pokemon.csv', 'w', encoding = 'utf-8') as f:
    data.to_csv(f, index=False)